# Screenshots of test patterns

**Copyright 2026 Martin Köhler**

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

KLayout screenshots of the test patterns of the appendix (KPEX/2.5D vs MAGIC), like the other test pattern figures:
black background, dotted grid, scale bar and the layer colors of the PDK, with the extracted areas marked in red (frame and hatch).

The figure includes (`figures/test_patterns/*.qmd`) write the capacitances onto them.

In [ ]:
import glob
import os
from typing import *
import xml.etree.ElementTree as ET

import klayout.db as kdb
import klayout.lay as klay

# NOTE: the klayout-pex checkout, next to this repository by default
KPEX_DIR = os.path.abspath(os.environ.get('KPEX_DIR', os.path.join('..', '..', 'klayout-pex')))
OUTPUT_DIR = os.path.abspath(os.path.join('..', 'figures', 'test_patterns'))
# NOTE: the display settings of the KLayout GUI the other test pattern figures were made with
KLAYOUTRC = os.path.abspath('test_pattern_screenshots.klayoutrc')


def load_technology(pdk: str):
    """
    The KLayout technology of KPEX for the PDK, whose layer properties give the colors
    (the layout view takes the only technology there is)
    """
    kdb.Technology.clear_technologies()
    tech = kdb.Technology.create_technology(pdk)
    # NOTE: the only one of the PDK, not always named after it (e.g. sg13g2.lyt of ihp-sg13g2)
    tech.load(glob.glob(os.path.join(KPEX_DIR, 'pdk', pdk, 'libs.tech', 'kpex', '*.lyt'))[0])


def apply_klayoutrc(lv: klay.LayoutView, path: str):
    """
    Applies the settings of a klayoutrc (<config><key>value</key>…</config>) to the layout view
    """
    for setting in ET.parse(path).getroot():
        lv.set_config(setting.tag, (setting.text or '').strip())


def screenshot(pdk: str,
               pattern: str,
               aspect: str,
               view_box: kdb.DBox,
               marked_boxes: List[kdb.DBox | kdb.DPolygon],
               pixels_per_um: float = 17.6,
               marker_color: int = 0xff0000,
               marker_dither_pattern: int = 9,
               hidden_layers: Sequence[Tuple[int, int]] = (),
               texts_to_the_left: Collection[str] = (),
               edit_layout: Optional[Callable[[kdb.Layout], None]] = None) -> str:
    """
    Saves the screenshot <pattern>_-_<aspect>.png of the test pattern, with the marked boxes (in µm)

    :param marked_boxes: boxes or polygons
    :param marker_dither_pattern: of the marked boxes, by default lightly right-hatched, 1 is hollow
    :param hidden_layers: GDS pairs of layers that aren't shown (e.g. markers, whose fill hides the other shapes)
    :param texts_to_the_left: labels to draw to the left of their position, apart from a neighbor to the right of them
    :param edit_layout: changes the layout for this view only (e.g. to leave out what is above or below a cell)

    :return: the path of the screenshot
    """
    load_technology(pdk)
    lv = klay.LayoutView(editable=True)  # NOTE: for changes of the layout, this view only
    lv.load_layout(os.path.join(KPEX_DIR, 'testdata', 'designs', pdk, 'test_patterns', f"{pattern}.gds.gz"))
    layout = lv.cellview(0).layout()
    if edit_layout is not None:
        edit_layout(layout)
    for cell in layout.each_cell():
        for layer_index in layout.layer_indexes():
            for shape in cell.shapes(layer_index).each(kdb.Shapes.STexts):
                if shape.text_string in texts_to_the_left:
                    shape.text_halign = kdb.Text.HAlignRight.to_i()
    lv.max_hier()
    apply_klayoutrc(lv, KLAYOUTRC)
    lv.zoom_box(view_box)
    for layer_properties in lv.each_layer():
        if (layer_properties.source_layer, layer_properties.source_datatype) in hidden_layers:
            layer_properties.visible = False

    markers = []  # NOTE: the markers are shown as long as they exist
    for box in marked_boxes:
        m = klay.Marker(lv)
        m.set(box)
        m.color = marker_color
        m.frame_color = marker_color
        m.dither_pattern = marker_dither_pattern
        m.line_width = 1
        markers.append(m)

    # NOTE: like the screenshots of the GUI on a HiDPI display: 2x pixels (fonts, lines),
    #       about as many pixels per µm as the other test pattern figures
    path = os.path.join(OUTPUT_DIR, f"{pattern}_-_{aspect}.png")
    lv.save_image_with_options(path,
                               width=round(view_box.width() * pixels_per_um),
                               height=round(view_box.height() * pixels_per_um),
                               resolution=0.5)
    return path

## Example: overlap_shield_same_net_*

Plates of 100 µm x 100 µm, `li1` and `met1` of net `A` under `met2` of net `B`, `met1` covering all or the left half of `li1`.
The marked areas are the overlaps of `met2` with `met1` and `li1`.
The view has 20 µm below the plates, for the scale bar and its text in the XXXLarge font, and 10 µm at the other sides.

In [ ]:
screenshot('sky130A', 'overlap_shield_same_net_100um_x_100um_li1_m1_m2', 'overlap',
           view_box=kdb.DBox(-10, -20, 110, 110),
           marked_boxes=[kdb.DBox(0, 0, 100, 100)])

In [ ]:
screenshot('sky130A', 'overlap_shield_same_net_half_100um_x_100um_li1_m1_m2', 'overlap',
           view_box=kdb.DBox(-10, -20, 110, 110),
           marked_boxes=[kdb.DBox(0, 0, 50, 100), kdb.DBox(50, 0, 100, 100)])

## Example: plate_20um_x_20um_li1_over_*

A `li1` plate of 20 µm x 20 µm of net `A` over the middle of a tap of 40 µm x 40 µm, an n-tap in an nwell `W` or a p-tap tied to `VSS`.
The marked area is the overlap of `li1` with the tap.
The view is smaller than the one of the plates above, so more pixels per µm keep the screenshots as wide (2112 px), as the figures show them at the same width.

In [ ]:
screenshot('sky130A', 'plate_20um_x_20um_li1_over_ntap', 'overlap',
           view_box=kdb.DBox(-5, -12, 45, 45),
           marked_boxes=[kdb.DBox(10, 10, 30, 30)],
           pixels_per_um=42.24)

In [ ]:
screenshot('sky130A', 'plate_20um_x_20um_li1_over_ptap', 'overlap',
           view_box=kdb.DBox(-5, -12, 45, 45),
           marked_boxes=[kdb.DBox(10, 10, 30, 30)],
           pixels_per_um=42.24)

## Example: exclusion_*

The patterns of the exclusion markers of `KPEX/2.5D`, with the marked frames being the markers (`NoRCX.drawing`).
The views have the wires and the markers, with some space below for the scale bar.


In [ ]:
EXCLUSION_WIDTH_PX = 2112  # NOTE: the figures show them at the same width
NORCX_DRAWING = (148, 0)
IND_DRAWING = (27, 0)
TEXT_DRAWING = (63, 0)  # NOTE: the parameters and names that the pcells write over their devices


def exclusion_screenshot(pattern: str,
                         view_box: kdb.DBox,
                         hidden_layers: Sequence[Tuple[int, int]] = (),
                         texts_to_the_left: Collection[str] = ()) -> str:
    """
    A screenshot with the polygons of the markers as hollow frames, as there are shapes within them,
    in a color no layer of the PDK has, but not the layers of the markers, whose fill hides these shapes
    """
    layout = kdb.Layout()
    layout.read(os.path.join(KPEX_DIR, 'testdata', 'designs', 'ihp-sg13g2', 'test_patterns', f"{pattern}.gds.gz"))
    # NOTE: layer() as a pattern needn't have markers, find_layer() would be None then, which is taken as the first layer
    marker_shapes = kdb.Region(layout.top_cell().begin_shapes_rec(layout.layer(*NORCX_DRAWING))).merged()
    return screenshot('ihp-sg13g2', pattern, 'layout', view_box=view_box,
                      marked_boxes=[polygon.to_dtype(layout.dbu) for polygon in marker_shapes.each()],
                      pixels_per_um=EXCLUSION_WIDTH_PX / view_box.width(),
                      marker_color=0x00ffff, marker_dither_pattern=1,
                      hidden_layers=[NORCX_DRAWING, IND_DRAWING, TEXT_DRAWING, *hidden_layers],
                      texts_to_the_left=texts_to_the_left)


In [ ]:
exclusion_screenshot('exclusion_wire_through', kdb.DBox(-25, -15, 45, 12))


In [ ]:
exclusion_screenshot('exclusion_plate_neighbor', kdb.DBox(-25, -12, 45, 28))


In [ ]:
exclusion_screenshot('exclusion_stubs', kdb.DBox(-13, -9, 13, 14))


In [ ]:
# NOTE: Metal3.text has the same labels as Metal3.pin, and the pcell writes its name on HeatTrans too
exclusion_screenshot('exclusion_nmos_diode2', kdb.DBox(-2.5, -2.3, 2, 2.6), hidden_layers=[(30, 25), (51, 0)])


In [ ]:
# NOTE: IND.text has the labels of the leads once more, at the edge of IND
exclusion_screenshot('exclusion_inductor2_wire_under', kdb.DBox(-58, -62, 58, 86),
                     hidden_layers=[(27, 25)], texts_to_the_left=['LA'])


In [ ]:
exclusion_screenshot('exclusion_coil_guard_ring', kdb.DBox(-68, -36, 68, 90))


In [ ]:
exclusion_screenshot('cut_coil', kdb.DBox(-68, -36, 68, 100), texts_to_the_left=['VDD'])


In [ ]:
exclusion_screenshot('exclusion_hairpin', kdb.DBox(-33, -9, 13, 14))


## Example: cut_block_with_mos

A cell cut out with `--cut`: the layout as drawn, the cell alone and the rest of the layout, at the same scale.
The figure includes (`figures/test_patterns/cut_block_with_mos_-_*.qmd`) frame the cell and write the ports onto them.


In [ ]:
CUT_VIEW_BOX = kdb.DBox(-7, -3.2, 12, 2.6)


def without_the_second_labels_of_the_cell(layout: kdb.Layout):
    # NOTE: Metal3.text of the cell has the same labels as its Metal3.pin
    layout.cell('block').shapes(layout.layer(30, 25)).clear()


def only_the_cell(layout: kdb.Layout):
    without_the_second_labels_of_the_cell(layout)
    for layer_index in layout.layer_indexes():
        layout.top_cell().shapes(layer_index).clear()


def without_the_cell(layout: kdb.Layout):
    for instance in list(layout.top_cell().each_inst()):
        instance.delete()


for aspect, edit_layout in (('layout', without_the_second_labels_of_the_cell),
                            ('cell', only_the_cell),
                            ('extracted', without_the_cell)):
    screenshot('ihp-sg13g2', 'cut_block_with_mos', aspect, view_box=CUT_VIEW_BOX, marked_boxes=[],
               pixels_per_um=EXCLUSION_WIDTH_PX / CUT_VIEW_BOX.width(),
               hidden_layers=[TEXT_DRAWING, (51, 0)], edit_layout=edit_layout)
